# Datasets Sources
- https://www.kaggle.com/competitions/demand-forecasting-kernels-only/models
- https://www.kaggle.com/competitions/m5-forecasting-accuracy/data


# Model
- LSTM, GRU, RNN comparison

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import copy
import random
import time
import json
from pathlib import Path
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

In [ ]:
# Hyperparameter
BATCH_SIZE = 64
EPOCHS = 15
learning_rate = 1e-3
SEED = 7
LOOKBACK = 60
HORIZON = 28                 # ตั้งเป็น 1 ได้เพื่อดูอาการทำนายตามหลังแบบวันต่อวัน
HIDDEN_SIZE = 32
MODEL_NAMES = ['LSTM', 'GRU', 'RNN']  # หรือ ['GRU'] เพื่อเริ่มรันตัวเดียว
TRAIN_RATIO = 0.70
VAL_RATIO = 0.15

ROOT = Path('/content/drive/MyDrive/demand-forecasting')
RUN_NAME = 'm5_simple_v1'  # เปลี่ยนชื่อเมื่อทำการทดลองใหม่
OUTPUT_DIR = ROOT / 'outputs' / 'm5' / RUN_NAME
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything(SEED)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Device:', device, '| PyTorch:', torch.__version__)

In [ ]:
# Build M5 Time-Series Pipeline
path = ROOT / 'data/m5/sales_train_evaluation.csv'
selected = pd.read_csv(path, nrows=1).iloc[0]
day_cols = sorted([c for c in selected.index if c.startswith('d_')],
                  key=lambda c: int(c[2:]))
calendar = pd.read_csv(ROOT / 'data/m5/calendar.csv', usecols=['d', 'date'])
raw = pd.DataFrame({
    'date': pd.to_datetime(calendar.set_index('d').loc[day_cols, 'date']).to_numpy(),
    'sales': selected[day_cols].to_numpy(dtype=np.float32),
})
series_name = str(selected['id'])

sales = raw['sales'].to_numpy(dtype=np.float32)
dates = pd.DatetimeIndex(raw['date'])
n = len(sales)
train_end = int(n * TRAIN_RATIO)
val_end = int(n * (TRAIN_RATIO + VAL_RATIO))

split_table = pd.DataFrame([
    {'split':'Train', 'start':str(dates[0].date()), 'end':str(dates[train_end-1].date())},
    {'split':'Validation', 'start':str(dates[train_end].date()), 'end':str(dates[val_end-1].date())},
    {'split':'Test', 'start':str(dates[val_end].date()), 'end':str(dates[-1].date())},
])
print(series_name, '| days:', n)
display(raw.head())
display(split_table)

In [ ]:
# Explore the time series
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(dates, sales, linewidth=0.8)
ax.axvline(dates[train_end], color='orange', linestyle='--', label='Validation starts')
ax.axvline(dates[val_end], color='red', linestyle='--', label='Test starts')
ax.set(title=series_name, xlabel='Date', ylabel='Units sold')
ax.legend()
plt.show()

In [ ]:
# Scaling and normalization (fit on Train ONLY)
diff = np.full(n, np.nan, dtype=np.float32)
diff[1:] = sales[1:] - sales[:-1]
sales_mu = float(sales[:train_end].mean())
sales_sigma = max(float(sales[:train_end].std()), 1e-6)
diff_mu = float(diff[1:train_end].mean())
diff_sigma = max(float(diff[1:train_end].std()), 1e-6)

day_of_week = dates.dayofweek.to_numpy()
day_of_year = dates.dayofyear.to_numpy()
feature_names = ['sales_z', 'diff_z', 'dow_sin', 'dow_cos', 'doy_sin', 'doy_cos']
features = np.column_stack([
    (sales - sales_mu) / sales_sigma,
    (diff - diff_mu) / diff_sigma,
    np.sin(2*np.pi*day_of_week/7), np.cos(2*np.pi*day_of_week/7),
    np.sin(2*np.pi*day_of_year/365.25), np.cos(2*np.pi*day_of_year/365.25),
]).astype(np.float32)
targets = ((diff - diff_mu) / diff_sigma).astype(np.float32)
INPUT_SIZE = features.shape[1]

class SalesDataset(Dataset):
    def __init__(self, origins):
        self.origins = np.asarray(list(origins), dtype=np.int64)

    def __len__(self):
        return len(self.origins)

    def __getitem__(self, index):
        t = int(self.origins[index])  # t คือวันแรกที่ต้องการทำนาย
        x = features[t-LOOKBACK:t]   # ล่าสุดที่เห็นคือ t-1
        y = targets[t:t+HORIZON]
        anchor = sales[t-1]          # ใช้แปลง diff กลับเป็นยอดขาย
        actual_sales = sales[t:t+HORIZON]
        return (torch.from_numpy(x.copy()), torch.from_numpy(y.copy()),
                torch.tensor(anchor), torch.from_numpy(actual_sales.copy()), t)

train_set = SalesDataset(range(LOOKBACK+1, train_end-HORIZON+1))
# Validation/Test ประเมินใหม่ทุก H วัน: nonoverlapping rolling-origin blocks
# แต่ละ origin ใช้เฉพาะข้อมูลที่ทราบก่อนเริ่ม block นั้น
val_set = SalesDataset(range(train_end, val_end-HORIZON+1, HORIZON))
test_set = SalesDataset(range(val_end, n-HORIZON+1, HORIZON))
print('Windows:', len(train_set), len(val_set), len(test_set))
print('Features:', feature_names)

In [ ]:
# Build Dataloader


In [ ]:
# Build Model


In [ ]:
# Inspect shapes

In [ ]:
# Define loss and optimizer

In [ ]:
# Training and Validating

In [ ]:
# Plot learning curves

In [ ]:
# Evaluation